Task 01 | Nested Transaction Analyzer

Concepts: Data structures, sorting, comprehensions

Problem: Given a list of transaction dictionaries with keys id, customer, category, amount, and status, produce a report containing each customer’s successful spending, top category, and rank by spending. Ignore failed transactions and reject duplicate transaction IDs.

Requirements: Return a list sorted by spending descending, then customer name ascending. Customers with no successful transactions must appear with total 0. Do not use pandas.

Sample / expected behavior: Input: [{id:1,customer:"Ali",category:"Food",amount:500,status:"success"},{id:2,customer:"Ali",category:"Travel",amount:200,status:"failed"},{id:3,customer:"Sara",category:"Food",amount:700,status:"success"}]. Output totals: Sara=700, Ali=500.

In [ ]:

def analyze_transactions(transactions: list) -> list:
    seen_ids = set()
    unique_transactions = []
    all_customers = set()

    for tx in transactions:
        all_customers.add(tx['customer'])
        if tx['id'] in seen_ids:
            continue
        seen_ids.add(tx['id'])
        unique_transactions.append(tx)

    customer_metrics = {
        name: {'total_spending': 0, 'categories': {}} 
        for name in all_customers
    }

    for tx in unique_transactions:
        if tx['status'] == 'success':
            customer = tx['customer']
            amount = tx['amount']
            category = tx['category']
            
            customer_metrics[customer]['total_spending'] += amount
            cat_map = customer_metrics[customer]['categories']
            cat_map[category] = cat_map.get(category, 0) + amount

    report = []
    for customer, data in customer_metrics.items():
        total = data['total_spending']
        categories = data['categories']
        
        if categories:
            max_spend = max(categories.values())
            top_categories = [cat for cat, spend in categories.items() if spend == max_spend]
            top_category = min(top_categories)  # Lexicographically earliest
        else:
            top_category = None

        report.append({
            "customer": customer,
            "total_spending": total,
            "top_category": top_category
        })

    report.sort(key=lambda x: (-x['total_spending'], x['customer']))

    for index, record in enumerate(report):
        record['rank'] = index + 1

    return report


sample_input = [
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
    {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
    {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
    {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"}
]

output_report = analyze_transactions(sample_input)
for row in output_report:
    print(row)




{'customer': 'Sara', 'total_spending': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total_spending': 500, 'top_category': 'Food', 'rank': 2}


Task 02 | Longest Valid Window

Concepts: Sliding window, dictionaries

Problem: Write longest_unique_window(text) to find the longest contiguous substring with no repeated characters. Return the substring, start index, and length.

Requirements: Target O(n) time. On equal-length windows, select the earliest. Treat spaces and punctuation as characters.

Sample / expected behavior: Input: "abcabcbb" -> ("abc", 0, 3); "" -> ("", 0, 0).

In [ ]:
def longest_unique_window(text: str) -> tuple[str, int, int]:
    
    if not text:
        return ("", 0, 0)
        
    char_map = {}
    start = 0
    max_len = 0
    best_start = 0
    
    for end, char in enumerate(text):
        # If character is seen inside the current window, shrink window by jumping past its last location
        if char in char_map and char_map[char] >= start:
            start = char_map[char] + 1
            
        char_map[char] = end
        
        
        current_len = end - start + 1
        
        
        if current_len > max_len:
            max_len = current_len
            best_start = start
            
    return (text[best_start:best_start + max_len], best_start, max_len)


Task 03 | Recursive JSON Path Explorer

Concepts: Recursion, nested lists/dictionaries

Problem: Implement find_paths(data, target) that recursively searches nested dictionaries and lists and returns all paths whose value equals target.

Requirements: Use dot notation for dictionary keys and [index] for list positions. Preserve traversal order; do not treat bool True as numeric 1.

Sample / expected behavior: Input: {"a":[{"x":5},5],"b":{"y":5}}, target=5 -> ["a[0].x", "a[1]", "b.y"]

In [2]:
def find_paths(data, target):
    """
    Recursively searches nested dictionaries and lists to find all paths
    where the value matches the target.
    
    Requirements:
    - Uses dot notation for dictionary keys.
    - Uses [index] for list positions.
    - Preserves traversal order.
    - Strictly distinguishes between boolean True/False and numeric 1/0.
    """
    def recurse(current, current_path):
        # Strict equality check to prevent True == 1 and False == 0 matching
        if type(current) is type(target) and current == target:
            paths.append(current_path)
            return

        if isinstance(current, dict):
            for key, value in current.items():
                # Format dictionary keys using dot notation
                next_path = f"{current_path}.{key}" if current_path else str(key)
                recurse(value, next_path)
                
        elif isinstance(current, list):
            for index, value in enumerate(current):
                # Format list indices using [index] notation
                next_path = f"{current_path}[{index}]" if current_path else f"[{index}]"
                recurse(value, next_path)

    paths = []
    recurse(data, "")
    return paths

if __name__ == "__main__":
    sample_data = {"a": [{"x": 5}, 5], "b": {"y": 5}}
    target_val = 5
    result = find_paths(sample_data, target_val)
    
    print(f"Input: {sample_data}, target={target_val}")
    print(f"Output: {result}")
    


Input: {'a': [{'x': 5}, 5], 'b': {'y': 5}}, target=5
Output: ['a[0].x', 'a[1]', 'b.y']


Task 04 | Dependency Execution Planner

Concepts: Graphs, sets, algorithms

Problem: Given tasks and prerequisite pairs (A,B) meaning A must finish before B, return a valid execution order.

Requirements: Detect cycles and raise ValueError with a clear message. For multiple available tasks, select alphabetically first. Reject unknown task names.

Sample / expected behavior: Tasks=["build","test","deploy"]; dependencies=[("build","test"),("test","deploy")] -> ["build","test","deploy"]

In [4]:
import heapq
from typing import List, Tuple

def plan_execution_order(tasks: List[str], dependencies: List[Tuple[str, str]]) -> List[str]:
    # 1. Initialize data structures
    task_set = set(tasks)
    in_degree = {task: 0 for task in tasks}
    adjacency_list = {task: [] for task in tasks}
    
    for u, v in dependencies:
        if u not in task_set:
            raise ValueError(f"Unknown task name in dependencies: '{u}'")
        if v not in task_set:
            raise ValueError(f"Unknown task name in dependencies: '{v}'")
            
        adjacency_list[u].append(v)
        in_degree[v] += 1
        
    available_tasks = [task for task in tasks if in_degree[task] == 0]
    heapq.heapify(available_tasks)
    
    execution_order = []
    
    while available_tasks:
        current_task = heapq.heappop(available_tasks)
        execution_order.append(current_task)
        
        for neighbor in adjacency_list[current_task]:
            in_degree[neighbor] -= 1
            if in_degree[neighbor] == 0:
                heapq.heappush(available_tasks, neighbor)
                
    if len(execution_order) != len(tasks):
        raise ValueError("Cyclic dependency detected! A valid execution order is impossible.")
        
    return execution_order


tasks_sample = ["build", "test", "deploy"]
deps_sample = [("build", "test"), ("test", "deploy")]
print(plan_execution_order(tasks_sample, deps_sample))
# Output: ['build', 'test', 'deploy']


['build', 'test', 'deploy']


Task 05 | Expression Validator and Evaluator

Concepts: Stacks, parsing, exception handling

Problem: Create a calculator for integer expressions containing +, -, *, /, parentheses, and whitespace. Apply standard operator precedence and parentheses.

Requirements: Do not use eval(), exec(), or third-party parsers. Handle unary minus, malformed expressions, and division by zero.

Sample / expected behavior: Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0.

In [6]:
import re

def evaluate_expression(expr: str) -> float:
    # Tokenize numbers, operators, and parentheses
    raw_tokens = re.findall(r'\d+|\+|\-|\*|\/|\(|\)', expr)
    if not raw_tokens:
        raise ValueError("Empty expression")

    tokens = []
    for i, t in enumerate(raw_tokens):
        if t == '-':
            is_unary = (i == 0) or (raw_tokens[i - 1] in '(+-*/')
            if is_unary:
                tokens.extend(['0', '-'])
                continue
        tokens.append(t)

    precedence = {'+': 1, '-': 1, '*': 2, '/': 2}
    output_queue = []
    operator_stack = []

    for t in tokens:
        if t.isdigit():
            output_queue.append(float(t))
        elif t in precedence:
            while (operator_stack and operator_stack[-1] in precedence and
                   precedence[operator_stack[-1]] >= precedence[t]):
                output_queue.append(operator_stack.pop())
            operator_stack.append(t)
        elif t == '(':
            operator_stack.append(t)
        elif t == ')':
            while operator_stack and operator_stack[-1] != '(':
                output_queue.append(operator_stack.pop())
            if not operator_stack:
                raise ValueError("Mismatched parentheses: missing '('")
            operator_stack.pop()  # Discard '('

    while operator_stack:
        op = operator_stack.pop()
        if op in ('(', ')'):
            raise ValueError("Mismatched parentheses")
        output_queue.append(op)

    eval_stack = []
    for token in output_queue:
        if isinstance(token, float):
            eval_stack.append(token)
        else:
            if len(eval_stack) < 2:
                raise ValueError("Malformed expression")
            b = eval_stack.pop()
            a = eval_stack.pop()
            if token == '+':
                eval_stack.append(a + b)
            elif token == '-':
                eval_stack.append(a - b)
            elif token == '*':
                eval_stack.append(a * b)
            elif token == '/':
                if b == 0:
                    raise ZeroDivisionError("Division by zero")
                eval_stack.append(a / b)

    if len(eval_stack) != 1:
        raise ValueError("Malformed expression")

    return eval_stack[0]


if __name__ == "__main__":
    sample = "-(3 + 2) * 4 + 18 / 3"
    result = evaluate_expression(sample)
    print(f'Input: "{sample}" -> {result}')


Input: "-(3 + 2) * 4 + 18 / 3" -> -14.0


Task 06 | Banking Ledger with OOP

Concepts: Classes, encapsulation, properties

Problem: Build Account and Bank classes. Support deposit, withdrawal, transfer, and account statements. Maintain immutable transaction records containing transaction ID, action, amount, and resulting balance.

Requirements: Reject nonpositive amounts, insufficient funds, and nonexistent accounts. A transfer must be atomic: both balances update or neither does.

Sample / expected behavior: Account A=1000, B=200; transfer 350 from A to B -> A=650, B=550.

In [7]:
import uuid
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional


@dataclass(frozen=True)
class TransactionRecord:
    """
    An immutable record of a single transaction.
    Using a frozen dataclass ensures the history cannot be modified after creation.
    """
    transaction_id: str
    action: str  # "Deposit", "Withdrawal", "Transfer Sent", "Transfer Received"
    amount: float
    resulting_balance: float
    timestamp: datetime = datetime.now()

    def __str__(self) -> str:
        return (
            f"[{self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}] "
            f"ID: {self.transaction_id} | {self.action}: ${self.amount:,.2f} | "
            f"Balance: ${self.resulting_balance:,.2f}"
        )


class Account:
    """
    Represents an individual bank account.
    Demonstrates encapsulation by keeping the balance and history private.
    """
    def __init__(self, account_id: str, owner_name: str, initial_balance: float = 0.0):
        if initial_balance < 0:
            raise ValueError("Initial balance cannot be negative.")
            
        self.account_id: str = account_id
        self.owner_name: str = owner_name
        self._balance: float = initial_balance  # Encapsulated attribute
        self._history: List[TransactionRecord] = []
        
        # Record the initial deposit if greater than 0
        if initial_balance > 0:
            self._record_transaction("Deposit (Initial)", initial_balance)

    @property
    def balance(self) -> float:
        """Read-only property for the balance."""
        return self._balance

    @property
    def statement(self) -> List[TransactionRecord]:
        """Returns a copy of the transaction history to preserve immutability."""
        return list(self._history)

    def _record_transaction(self, action: str, amount: float) -> None:
        """Internal helper to append immutable records to the history ledger."""
        record = TransactionRecord(
            transaction_id=str(uuid.uuid4())[:8],  # Compact ID for readability
            action=action,
            amount=amount,
            resulting_balance=self._balance
        )
        self._history.append(record)

    def deposit(self, amount: float) -> None:
        """Deposits a positive amount into the account."""
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        
        self._balance += amount
        self._record_transaction("Deposit", amount)

    def withdraw(self, amount: float) -> None:
        """Withdraws a positive amount if sufficient funds exist."""
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive.")
        if amount > self._balance:
            raise ValueError("Insufficient funds.")
        
        self._balance -= amount
        self._record_transaction("Withdrawal", amount)

    def _internal_transfer_out(self, amount: float) -> None:
        """Stage 1 of an atomic transfer: validation and local deduction."""
        if amount <= 0:
            raise ValueError("Transfer amount must be positive.")
        if amount > self._balance:
            raise ValueError("Insufficient funds for transfer.")
        
        self._balance -= amount

    def _internal_transfer_in(self, amount: float) -> None:
        """Stage 2 of an atomic transfer: local addition."""
        self._balance += amount


class Bank:
    """
    Manages accounts and coordinates transactions between them.
    Ensures operation atomicity and accounts validation.
    """
    def __init__(self):
        self._accounts: Dict[str, Account] = {}

    def create_account(self, name: str, initial_balance: float = 0.0) -> Account:
        """Generates a unique account, adds it to the registry, and returns it."""
        account_id = f"ACC-{len(self._accounts) + 1:03d}"
        new_account = Account(account_id, name, initial_balance)
        self._accounts[account_id] = new_account
        return new_account

    def get_account(self, account_id: str) -> Account:
        """Fetches an account or rejects if it does not exist."""
        if account_id not in self._accounts:
            raise KeyError(f"Account {account_id} not found.")
        return self._accounts[account_id]

    def transfer(self, from_id: str, to_id: str, amount: float) -> None:
        """
        Executes an atomic transfer between two accounts.
        Guarantees that both update successfully or neither does.
        """
        source = self.get_account(from_id)
        destination = self.get_account(to_id)

        if source == destination:
            raise ValueError("Cannot transfer funds to the same account.")

        # --- ATOMIC TRANSACTION BLOCK ---
        # 1. Stage the withdrawal (will fail early if funds are insufficient)
        source._internal_transfer_out(amount)
        
        try:
            # 2. Stage the deposit
            destination._internal_transfer_in(amount)
        except Exception as e:
            # 3. Rollback source balance if destination injection fails unexpectedly
            source._internal_transfer_in(amount)
            raise RuntimeError(f"Transfer failed and was rolled back safely. Error: {e}")
        
        # 4. Commit records to both ledgers only after balances are safely updated
        source._record_transaction(f"Transfer Sent to {to_id}", amount)
        destination._record_transaction(f"Transfer Received from {from_id}", amount)


# ==========================================
# VERIFICATION & SAMPLE BEHAVIOR
# ==========================================
if __name__ == "__main__":
    bank = Bank()

    # Create the sample accounts
    account_a = bank.create_account("Alice", 1000.0)
    account_b = bank.create_account("Bob", 200.0)

    print("--- Initial State ---")
    print(f"Account A ({account_a.account_id}) Balance: ${account_a.balance}")
    print(f"Account B ({account_b.account_id}) Balance: ${account_b.balance}\n")

    # Perform sample transfer requirement: 350 from A to B
    print("Executing transfer of $350 from A to B...")
    bank.transfer(account_a.account_id, account_b.account_id, 350.0)
    
    print("\n--- State After Transfer ---")
    print(f"Account A ({account_a.account_id}) Balance: ${account_a.balance} (Expected: 650.0)")
    print(f"Account B ({account_b.account_id}) Balance: ${account_b.balance} (Expected: 550.0)\n")

    # Test error rejection constraints
    print("--- Testing Rejections & Atomicity ---")
    
    try:
        print("Attempting to transfer negative amount (-$50)...")
        bank.transfer(account_a.account_id, account_b.account_id, -50.0)
    except ValueError as e:
        print(f"Rejected successfully: {e}")

    try:
        print("Attempting to transfer more than available balance ($10,000)...")
        bank.transfer(account_a.account_id, account_b.account_id, 10000.0)
    except ValueError as e:
        print(f"Rejected successfully: {e}")
        print(f"Atomicity confirmation - Account A balance remains: ${account_a.balance}")

    try:
        print("Attempting transfer to a fake account...")
        bank.transfer(account_a.account_id, "ACC-999", 50.0)
    except KeyError as e:
        print(f"Rejected successfully: Account code {e} does not exist.")
        print(f"Atomicity confirmation - Account A balance remains: ${account_a.balance}\n")

    # Display statements
    print("--- Account A Full Statement Ledger ---")
    for row in account_a.statement:
        print(row)


--- Initial State ---
Account A (ACC-001) Balance: $1000.0
Account B (ACC-002) Balance: $200.0

Executing transfer of $350 from A to B...

--- State After Transfer ---
Account A (ACC-001) Balance: $650.0 (Expected: 650.0)
Account B (ACC-002) Balance: $550.0 (Expected: 550.0)

--- Testing Rejections & Atomicity ---
Attempting to transfer negative amount (-$50)...
Rejected successfully: Transfer amount must be positive.
Attempting to transfer more than available balance ($10,000)...
Rejected successfully: Insufficient funds for transfer.
Atomicity confirmation - Account A balance remains: $650.0
Attempting transfer to a fake account...
Rejected successfully: Account code 'Account ACC-999 not found.' does not exist.
Atomicity confirmation - Account A balance remains: $650.0

--- Account A Full Statement Ledger ---
[2026-11-07 15:23:39] ID: 4d45b422 | Deposit (Initial): $1,000.00 | Balance: $1,000.00
[2026-11-07 15:23:39] ID: 4eeaecc2 | Transfer Sent to ACC-002: $350.00 | Balance: $650.00


Task 07 | Employee Payroll Engine

Concepts: Inheritance, polymorphism, abstract classes

Problem: Define abstract Employee with calculate_pay(). Implement SalariedEmployee, HourlyEmployee, and CommissionEmployee with different pay formulas. Generate a unified payroll summary.

Requirements: Hourly overtime above 40 hours pays 1.5x. Commission employees receive base + sales*rate. Prevent direct instantiation of Employee.

Sample / expected behavior: Hourly rate=20, hours=45 -> 950; Commission base=500, sales=4000, rate=.10 -> 900.

In [8]:
from abc import ABC, abstractmethod

class Employee(ABC):
    """
    Abstract base class representing a generic employee.
    Cannot be directly instantiated.
    """
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def calculate_pay(self) -> float:
        """Abstract method to be overridden by subclasses."""
        pass


class SalariedEmployee(Employee):
    """Employee paid a fixed weekly/monthly salary."""
    def __init__(self, name: str, salary: float):
        super().__init__(name)
        self.salary = salary

    def calculate_pay(self) -> float:
        return self.salary


class HourlyEmployee(Employee):
    """Employee paid by the hour with overtime rules (1.5x after 40 hours)."""
    def __init__(self, name: str, hourly_rate: float, hours_worked: float):
        super().__init__(name)
        self.hourly_rate = hourly_rate
        self.hours_worked = hours_worked

    def calculate_pay(self) -> float:
        if self.hours_worked <= 40:
            return self.hours_worked * self.hourly_rate
        
        regular_pay = 40 * self.hourly_rate
        overtime_hours = self.hours_worked - 40
        overtime_pay = overtime_hours * (self.hourly_rate * 1.5)
        return regular_pay + overtime_pay


class CommissionEmployee(Employee):
    """Employee paid a base salary plus a percentage of sales."""
    def __init__(self, name: str, base_salary: float, sales: float, commission_rate: float):
        super().__init__(name)
        self.base_salary = base_salary
        self.sales = sales
        self.commission_rate = commission_rate

    def calculate_pay(self) -> float:
        return self.base_salary + (self.sales * self.commission_rate)


def generate_payroll_summary(employees: list[Employee]) -> None:
    """Polymorphically processes a list of employees and outputs a summary."""
    print("=== UNIFIED PAYROLL SUMMARY ===")
    total_payout = 0.0
    
    for emp in employees:
        pay = emp.calculate_pay()
        total_payout += pay
        print(f"Employee: {emp.name:<15} | Type: {type(emp).__name__:<18} | Pay: ${pay:,.2f}")
        
    print("=" * 31)
    print(f"Total Company Payout: ${total_payout:,.2f}\n")


# --- Demonstration & Verification ---
if __name__ == "__main__":
    # 1. Enforce Abstraction Rule (Will throw TypeError)
    try:
        ghost = Employee("Generic Object")
    except TypeError as e:
        print(f"Success: Cannot instantiate Employee directly -> Error: {e}\n")

    # 2. Instantiate Concrete Subclasses
    emp1 = SalariedEmployee("Alice", 1500.00)
    emp2 = HourlyEmployee("Bob", hourly_rate=20, hours_worked=45)       # Expected: 950
    emp3 = CommissionEmployee("Charlie", base_salary=500, sales=4000, commission_rate=0.10) # Expected: 900

    # 3. Polymorphically process payroll
    staff_payroll = [emp1, emp2, emp3]
    generate_payroll_summary(staff_payroll)


Success: Cannot instantiate Employee directly -> Error: Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'

=== UNIFIED PAYROLL SUMMARY ===
Employee: Alice           | Type: SalariedEmployee   | Pay: $1,500.00
Employee: Bob             | Type: HourlyEmployee     | Pay: $950.00
Employee: Charlie         | Type: CommissionEmployee | Pay: $900.00
Total Company Payout: $3,350.00



Task 08 | Cache Decorator with Expiration

Concepts: Decorators, closures, functools, time

Problem: Implement @ttl_cache(seconds) that caches results for repeated positional and keyword arguments and recomputes after expiration. Track cache hits and misses.

Requirements: Preserve function metadata with functools.wraps. Handle equivalent keyword argument order; document behavior for unhashable inputs. Use time.monotonic().

Sample / expected behavior: A call with x=2,y=3 followed by y=3,x=2 before expiry should return a cache hit

In [13]:
from functools import wraps
import time

def ttl_cache(seconds: float):
    """
    A decorator that caches function results for a given Time-To-Live (TTL) duration.
    
    Tracks cache hits and misses. Handles equivalent keyword arguments regardless
    of order. Preserves function metadata.
    
    Unhashable Input Behavior:
        If any positional or keyword argument is unhashable (e.g., lists, dicts, sets), 
        the cache key cannot be generated. In this case, the decorator bypasses the 
        cache entirely, executes the underlying function, and increments the 'misses' counter.
    """
    def decorator(func):
        # The cache dictionary will store: cache_key -> (result, expiry_time)
        cache = {}
        stats = {"hits": 0, "misses": 0}
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            sorted_kwargs = tuple(sorted(kwargs.items()))
            cache_key = (args, sorted_kwargs)
            
            try:
                current_time = time.monotonic()
                
                if cache_key in cache:
                    result, expiry_time = cache[cache_key]
                    if current_time < expiry_time:
                        stats["hits"] += 1
                        return result
                    else:
                        del cache[cache_key]
                        
                stats["misses"] += 1
                result = func(*args, **kwargs)
                cache[cache_key] = (result, current_time + seconds)
                return result
                
            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)
                
        wrapper.cache_stats = lambda: stats
        wrapper.cache_clear = lambda: cache.clear()
        
        return wrapper
    return decorator



##########################################################################################################3

@ttl_cache(seconds=2.0)
def complex_calc(x, y):
    return x + y

print(complex_calc(x=2, y=3))  # Returns 5
print(complex_calc.cache_stats())  # {'hits': 0, 'misses': 1}

print(complex_calc(y=3, x=2))  # Returns 5
print(complex_calc.cache_stats())  # {'hits': 1, 'misses': 1}


@ttl_cache(seconds=2.0)
def process_list(lst):
    return sum(lst)

print(process_list([1, 2, 3]))
print(process_list.cache_stats())  # {'hits': 0, 'misses': 1}


5
{'hits': 0, 'misses': 1}
5
{'hits': 1, 'misses': 1}
6
{'hits': 0, 'misses': 1}


Task 09 | Streaming Log Inspector

Concepts: Generators, iterators, file handling

Problem: Write a generator read_errors(path) that streams a log file line by line, yielding (line_number, timestamp, message) only for ERROR records. Build summarize_errors() to count errors by date.

Requirements: Do not load the whole file into memory. Skip malformed lines while tracking their count. Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message.

Sample / expected behavior: Input line: 2026-11-07 10:30:00 | ERROR | Connection lost -> (1,"2026-11-07 10:30:00","Connection lost").

In [ ]:
from collections import Counter

def read_errors(path, stats=None):
    """
    Generator that streams a log file line by line, yielding 
    (line_number, timestamp, message) only for ERROR records.
    
    Args:
        path (str): Path to the log file.
        stats (dict, optional): Mutable dictionary to track malformed line counts.
    """
    if stats is None:
        stats = {'malformed': 0}
        
    with open(path, 'r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            
            # Expecting format: YYYY-MM-DD HH:MM:SS | LEVEL | message
            parts = line.split(' | ', 2)
            if len(parts) != 3:
                stats['malformed'] += 1
                continue
                
            timestamp, level, message = parts
            
            if level.strip() == 'ERROR':
                yield (line_number, timestamp.strip(), message.strip())

def summarize_errors(path):
    """
    Counts ERROR records by date and tracks malformed lines.
    
    Returns:
        tuple: (date_counts (dict), malformed_count (int))
    """
    stats = {'malformed': 0}
    date_counts = Counter()
    
    for line_number, timestamp, message in read_errors(path, stats):
    
        date_str = timestamp.split()[0]
        date_counts[date_str] += 1
        
    return dict(date_counts), stats['malformed']


Task 10 | CSV Inventory Reconciliation

Concepts: CSV, file handling, validation

Problem: Read inventory.csv (sku,name,quantity) and movements.csv (sku,type,units). Apply IN and OUT movements in order, then write updated_inventory.csv and rejected_movements.csv.

Requirements: Reject unknown SKUs, negative or zero units, invalid movement types, and any OUT causing negative stock. Continue processing after rejected rows.

Sample / expected behavior: SKU P01 starts at 10; OUT 4, OUT 9, IN 2 -> final 8; second movement rejected.

In [5]:
import csv

def reconcile_inventory(inventory_path, movements_path, output_inv_path, output_rej_path):
    # Load and initialize current inventory
    inventory = {}
    inventory_names = {}
    
    with open(inventory_path, mode='r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            sku = row['sku'].strip()
            name = row['name'].strip()
            qty = int(row['quantity'])
            inventory[sku] = qty
            inventory_names[sku] = name

    rejected_movements = []

    # Process movements sequentially
    with open(movements_path, mode='r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for line_num, row in enumerate(reader, start=2):  # start=2 assuming line 1 is header
            sku = row.get('sku', '').strip()
            m_type = row.get('type', '').strip().upper()
            units_str = row.get('units', '').strip()

            # 1. Validation: Ensure units is a valid integer
            try:
                units = int(units_str)
            except ValueError:
                row['reason'] = "Invalid units format"
                rejected_movements.append(row)
                continue

            # 2. Validation: Units must be positive and greater than zero
            if units <= 0:
                row['reason'] = "Units must be greater than zero"
                rejected_movements.append(row)
                continue

            # 3. Validation: SKU must exist in inventory
            if sku not in inventory:
                row['reason'] = "Unknown SKU"
                rejected_movements.append(row)
                continue

            # 4. Process movement and check constraints
            if m_type == 'IN':
                inventory[sku] += units
            elif m_type == 'OUT':
                if inventory[sku] - units < 0:
                    row['reason'] = "Insufficient stock for OUT movement"
                    rejected_movements.append(row)
                    continue
                inventory[sku] -= units
            else:
                row['reason'] = "Invalid movement type (Must be IN or OUT)"
                rejected_movements.append(row)
                continue

    # Write updated inventory
    with open(output_inv_path, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['sku', 'name', 'quantity'])
        for sku in sorted(inventory.keys()):
            writer.writerow([sku, inventory_names[sku], inventory[sku]])

    # Write rejected movements
    with open(output_rej_path, mode='w', newline='', encoding='utf-8') as f:
        if rejected_movements:
            # Add 'reason' to the header fields dynamically
            fieldnames = list(rejected_movements[0].keys())
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rejected_movements)
        else:
            writer = csv.writer(f)
            writer.writerow(['sku', 'type', 'units', 'reason'])


Task 11 | Multi-File Student Gradebook

Concepts: File handling, dictionaries, exceptions

Problem: Merge students.csv (student_id,name) and scores.csv (student_id,subject,marks). Calculate per-student average, grade, and rank; export report.csv.

Requirements: Marks must be 0–100; reject unknown IDs and duplicate student-subject entries. Grades: A>=85, B>=70, C>=55, D>=40, otherwise F. Rank by average, then ID.

Sample / expected behavior: Student 101 with marks 90,80,85 -> average 85.0, grade A.

In [6]:
import csv
from collections import defaultdict

def generate_gradebook(students_file, scores_file, report_file):
    students = {}
    
    # 1. Load and validate students
    try:
        with open(students_file, mode='r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                student_id = row['student_id'].strip()
                name = row['name'].strip()
                if not student_id or not name:
                    print(f"Warning: Skipping invalid student row {row}")
                    continue
                students[student_id] = name
    except FileNotFoundError:
        print(f"Error: The file {students_file} was not found.")
        return
    except Exception as e:
        print(f"Error reading {students_file}: {e}")
        return

    # Track student-subject combinations to prevent duplicates
    # Structure: { student_id: { subject: marks } }
    student_scores = defaultdict(dict)
    
    # 2. Load and validate scores
    try:
        with open(scores_file, mode='r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                student_id = row['student_id'].strip()
                subject = row['subject'].strip()
                marks_str = row['marks'].strip()
                
                # Rule: Reject unknown IDs
                if student_id not in students:
                    print(f"ValidationError: Student ID {student_id} not found in roster. Skipping row.")
                    continue
                
                # Rule: Reject duplicate student-subject entries
                if subject in student_scores[student_id]:
                    print(f"ValidationError: Duplicate entry for Student {student_id} in {subject}. Skipping duplicate.")
                    continue
                
                # Rule: Validate marks range (0-100)
                try:
                    marks = float(marks_str)
                    if not (0 <= marks <= 100):
                        raise ValueError
                except ValueError:
                    print(f"ValidationError: Invalid marks '{marks_str}' for Student {student_id} in {subject}. Must be 0-100. Skipping.")
                    continue
                
                # Store valid score
                student_scores[student_id][subject] = marks
                
    except FileNotFoundError:
        print(f"Error: The file {scores_file} was not found.")
        return
    except Exception as e:
        print(f"Error reading {scores_file}: {e}")
        return

    # 3. Process metrics (Average, Grade)
    processed_students = []
    
    for student_id, name in students.items():
        scores = student_scores[student_id].values()
        
        if not scores:
            # Handle students with no recorded scores
            avg_score = 0.0
            grade = 'F'
        else:
            avg_score = round(sum(scores) / len(scores), 2)
            
            # Rule: Grading logic
            if avg_score >= 85:
                grade = 'A'
            elif avg_score >= 70:
                grade = 'B'
            elif avg_score >= 55:
                grade = 'C'
            elif avg_score >= 40:
                grade = 'D'
            else:
                grade = 'F'
                
        processed_students.append({
            'student_id': student_id,
            'name': name,
            'average': avg_score,
            'grade': grade
        })

    # 4. Rank by average (descending), then ID (ascending for deterministic tie-breaking)
    # Note: To sort student_id numerically if they are numbers, we safely parse them in the key
    def sorting_key(student):
        try:
            id_key = int(student['student_id'])
        except ValueError:
            id_key = student['student_id']
        return (-student['average'], id_key)

    processed_students.sort(key=sorting_key)

    # Assign ranks (handling absolute position indexing)
    for rank, student in enumerate(processed_students, start=1):
        student['rank'] = rank

    # 5. Export results to report.csv
    try:
        with open(report_file, mode='w', newline='', encoding='utf-8') as f:
            fieldnames = ['rank', 'student_id', 'name', 'average', 'grade']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            
            writer.writeheader()
            for student in processed_students:
                writer.writerow(student)
                
        print(f"Success: Report successfully exported to '{report_file}'.")
        
    except Exception as e:
        print(f"Error writing to {report_file}: {e}")

# Example Usage:
if __name__ == "__main__":
    # To run this, ensure 'students.csv' and 'scores.csv' exist in your directory.
    generate_gradebook('students.csv', 'scores.csv', 'report.csv')


Error: The file students.csv was not found.


Task 12 | Custom Iterable Range Toolkit

Concepts: Iterators, generators, dunder methods

Problem: Create a ReversibleRange(start,stop,step) class supporting iteration, reversed iteration, len(), and membership tests without constructing a list.

Requirements: Match Python range stop-exclusive semantics for positive and negative nonzero steps. Implement __iter__, __reversed__, __len__, and __contains__.

Sample / expected behavior: ReversibleRange(2,11,3) -> [2,5,8]; reversed -> [8,5,2]; 5 in object -> True.

In [7]:
import math

class ReversibleRange:
    """A custom iterable range toolkit that emulates Python's built-in range.
    
    Supports efficient iteration, reversed iteration, length checking, 
    and membership tests in O(1) time complexity without constructing a list.
    """
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("ReversibleRange() arg 3 must not be zero")
            
        self.start = start
        self.stop = stop
        self.step = step
        
        # Precompute length lazily but efficiently during initialization
        self._length = self._calculate_len()

    def _calculate_len(self):
        """Calculates the number of elements in the range in O(1) time."""
        if (self.step > 0 and self.start >= self.stop) or (self.step < 0 and self.start <= self.stop):
            return 0
        return math.ceil((self.stop - self.start) / self.step)

    def __len__(self):
        """Returns the number of elements in the range in O(1) time."""
        return self._length

    def __iter__(self):
        """Generates elements from start to stop (exclusive) using a generator."""
        current = self.start
        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        """Generates elements in reverse order without building an intermediate list."""
        if self._length == 0:
            return
        
        # Find the exact last element in the sequence
        last_element = self.start + (self._length - 1) * self.step
        current = last_element
        reverse_step = -self.step
        
        for _ in range(self._length):
            yield current
            current += reverse_step

    def __contains__(self, value):
        """Determines membership in O(1) mathematical time complexity."""
        if not isinstance(value, (int, float)):  # Strict type check similar to range
            return False
            
        # Out of bounds checks
        if self._length == 0:
            return False
            
        last_element = self.start + (self._length - 1) * self.step
        
        if self.step > 0:
            if value < self.start or value > last_element:
                return False
        else:
            if value > self.start or value < last_element:
                return False
                
        # Must be cleanly divisible by the step size from the start point
        return (value - self.start) % self.step == 0

    def __repr__(self):
        return f"ReversibleRange({self.start}, {self.stop}, {self.step})"


Task 13 | Booking Conflict Scheduler

Concepts: Datetime, sorting, interval algorithms

Problem: Implement a room booking scheduler. Each booking has room, start datetime, end datetime, and booking ID. Accept bookings only when they do not overlap existing bookings for the same room.

Requirements: Use datetime objects; reject end<=start and duplicate IDs. Adjacent bookings are allowed. Return conflicts for rejected requests.

Sample / expected behavior: Room A: 09:00–10:00 and 10:00–11:00 accepted; 09:30–10:30 rejected

In [8]:
from datetime import datetime
from typing import List, Dict, Tuple, Union, Optional

class Booking:
    def __init__(self, booking_id: str, room: str, start: datetime, end: datetime):
        self.id = booking_id
        self.room = room
        self.start = start
        self.end = end

    def __repr__(self):
        return f"Booking({self.id}: {self.room} {self.start.strftime('%H:%M')}-{self.end.strftime('%H:%M')})"


class RoomScheduler:
    def __init__(self):
        # Maps room name -> list of accepted Bookings (kept sorted by start time)
        self.rooms: Dict[str, List[Booking]] = {}
        # Tracks unique booking IDs globally to reject duplicates
        self.booking_ids = set()

    def add_booking(self, booking_id: str, room: str, start: datetime, end: datetime) -> Tuple[bool, Optional[Union[str, List[Booking]]]]:
        """
        Attempts to add a booking.
        Returns:
            Tuple[bool, reason_or_conflicts]
            - If accepted: (True, None)
            - If rejected due to format error: (False, "Reason string")
            - If rejected due to conflict: (False, List of conflicting Booking objects)
        """
        # Requirement: Reject duplicate IDs
        if booking_id in self.booking_ids:
            return False, "Duplicate booking ID."

        # Requirement: Reject end <= start
        if end <= start:
            return False, "Invalid duration: End time must be after start time."

        # Initialize room schedule if it doesn't exist
        if room not in self.rooms:
            self.rooms[room] = []

        active_schedule = self.rooms[room]
        conflicts = []

        # Find overlaps
        # Two intervals [s1, e1] and [s2, e2] overlap if max(s1, s2) < min(e1, e2)
        # (Using strict inequality because adjacent bookings like 10:00-11:00 and 11:00-12:00 are allowed)
        for existing in active_schedule:
            if max(start, existing.start) < min(end, existing.end):
                conflicts.append(existing)

        # If conflicts exist, reject and return them
        if conflicts:
            return False, conflicts

        # If no conflicts, accept the booking
        new_booking = Booking(booking_id, room, start, end)
        
        # Insert while maintaining sorted order by start time
        active_schedule.append(new_booking)
        active_schedule.sort(key=lambda b: b.start)
        
        # Register the global ID
        self.booking_ids.add(booking_id)
        
        return True, None


# --- Verification & Demo ---
if __name__ == "__main__":
    scheduler = RoomScheduler()
    
    # Base date helper
    day = lambda hour, minute: datetime(2026, 11, 7, hour, minute)

    print("--- Test Case 1: Sequential / Adjacent Bookings ---")
    # 09:00 - 10:00 -> Accepted
    res1, conf1 = scheduler.add_booking("B1", "Room A", day(9, 0), day(10, 0))
    print(f"B1 (09:00-10:00): Accepted={res1}")

    # 10:00 - 11:00 -> Accepted (Adjacent timestamps allowed)
    res2, conf2 = scheduler.add_booking("B2", "Room A", day(10, 0), day(11, 0))
    print(f"B2 (10:00-11:00): Accepted={res2}")

    print("\n--- Test Case 2: Overlapping Booking ---")
    # 09:30 - 10:30 -> Should be rejected because it overlaps both B1 and B2
    res3, conf3 = scheduler.add_booking("B3", "Room A", day(9, 30), day(10, 30))
    print(f"B3 (09:30-10:30): Accepted={res3} | Conflicts: {conf3}")

    print("\n--- Test Case 3: Edge Cases ---")
    # Invalid duration: End before start
    res4, conf4 = scheduler.add_booking("B4", "Room A", day(13, 0), day(12, 0))
    print(f"B4 (13:00-12:00): Accepted={res4} | Reason: {conf4}")

    # Duplicate Booking ID
    res5, conf5 = scheduler.add_booking("B1", "Room A", day(14, 0), day(15, 0))
    print(f"B5 (Duplicate ID 'B1'): Accepted={res5} | Reason: {conf5}")


--- Test Case 1: Sequential / Adjacent Bookings ---
B1 (09:00-10:00): Accepted=True
B2 (10:00-11:00): Accepted=True

--- Test Case 2: Overlapping Booking ---
B3 (09:30-10:30): Accepted=False | Conflicts: [Booking(B1: Room A 09:00-10:00), Booking(B2: Room A 10:00-11:00)]

--- Test Case 3: Edge Cases ---
B4 (13:00-12:00): Accepted=False | Reason: Invalid duration: End time must be after start time.
B5 (Duplicate ID 'B1'): Accepted=False | Reason: Duplicate booking ID.


Task 14 | Command-Line Contact Manager

Concepts: Modules, JSON, CRUD, validation

Problem: Build a menu-driven CLI contact manager with add, search, update, delete, list, and persistent JSON storage. Each contact has a unique ID, name, phone, and email.

Requirements: Use separate functions/modules for storage and business logic. Handle missing/corrupt JSON safely without silently deleting existing data. Search names case-insensitively.

Sample / expected behavior: Add ID=1, name="Ayesha"; restart application; search "aye" -> contact remains available

In [ ]:
import json
import os
import re

FILE_NAME = "contacts.json"

# ==========================================
# 1. STORAGE FUNCTIONS
# ==========================================

def load_contacts():
    """JSON file se contacts load karne ke liye"""
    if not os.path.exists(FILE_NAME):
        return {}
    
    try:
        with open(FILE_NAME, "r") as file:
            if os.path.getsize(FILE_NAME) == 0:
                return {}
            return json.load(file)
    except json.JSONDecodeError:
        print("\n[Error] JSON file kharab (corrupt) ho chuki hai! Purana data delete nahi kiya gaya.")
        return None  # Data ko silently delete hone se bachane ke liye None return karega

def save_contacts(contacts):
    """Contacts ko JSON file mein save karne ke liye"""
    if contacts is None:
        return
    with open(FILE_NAME, "w") as file:
        json.dump(contacts, file, indent=4)


# ==========================================
# 2. VALIDATION FUNCTIONS
# ==========================================

def validate_phone(phone):
    return bool(re.match(r"^\+?[\d\s-]{7,15}$", phone))

def validate_email(email):
    return bool(re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", email))


# ==========================================
# 3. CRUD BUSINESS LOGIC
# ==========================================

def add_contact(contacts):
    cid = input("Enter Unique ID: ").strip()
    if cid in contacts:
        print("[Error] Yeh ID pehle se mojud hai!")
        return

    name = input("Enter Name: ").strip()
    if not name:
        print("[Error] Name khali nahi ho sakta!")
        return

    phone = input("Enter Phone: ").strip()
    if not validate_phone(phone):
        print("[Error] Invalid Phone Number!")
        return

    email = input("Enter Email: ").strip()
    if not validate_email(email):
        print("[Error] Invalid Email Address!")
        return

    contacts[cid] = {"name": name, "phone": phone, "email": email}
    save_contacts(contacts)
    print(f"[Success] Contact '{name}' kamyabi se add ho gaya!")

def search_contacts(contacts):
    query = input("Search by name (case-insensitive): ").strip().lower()
    found = False
    
    print("\n--- Search Results ---")
    for cid, info in contacts.items():
        if query in info["name"].lower():
            print(f"ID: {cid} | Name: {info['name']} | Phone: {info['phone']} | Email: {info['email']}")
            found = True
            
    if not found:
        print("Koi contact nahi mila!")

def list_contacts(contacts):
    if not contacts:
        print("\nContact list khali hai!")
        return
    
    print("\n--- All Contacts ---")
    for cid, info in contacts.items():
        print(f"ID: {cid} | Name: {info['name']} | Phone: {info['phone']} | Email: {info['email']}")

def update_contact(contacts):
    cid = input("Enter ID to update: ").strip()
    if cid not in contacts:
        print("[Error] Is ID ka koi contact nahi mila!")
        return

    print("Leave blank if you don't want to change the value.")
    name = input(f"New Name ({contacts[cid]['name']}): ").strip()
    phone = input(f"New Phone ({contacts[cid]['phone']}): ").strip()
    email = input(f"New Email ({contacts[cid]['email']}): ").strip()

    if name:
        contacts[cid]['name'] = name
    if phone:
        if validate_phone(phone): contacts[cid]['phone'] = phone
        else: print("[Warning] Invalid phone, purana number hi rakha gaya hai.")
    if email:
        if validate_email(email): contacts[cid]['email'] = email
        else: print("[Warning] Invalid email, purani email hi rakhi gayi hai.")

    save_contacts(contacts)
    print("[Success] Contact update ho گیا!")

def delete_contact(contacts):
    cid = input("Enter ID to delete: ").strip()
    if cid in contacts:
        del contacts[cid]
        save_contacts(contacts)
        print("[Success] Contact delete ho gaya!")
    else:
        print("[Error] ID nahi mili!")


# ==========================================
# 4. MAIN INTERFACE
# ==========================================

def main():
    contacts = load_contacts()
    if contacts is None:
        print("Application ko band kiya ja raha hai taake data mehfooz rahe.")
        return

    while True:
        print("\n===== CONTACT MANAGER =====")
        print("1. Add Contact")
        print("2. Search Contact")
        print("3. List All Contacts")
        print("4. Update Contact")
        print("5. Delete Contact")
        print("6. Exit")
        
        choice = input("select your choice  (1-6): ").strip()

        if choice == "1": add_contact(contacts)
        elif choice == "2": search_contacts(contacts)
        elif choice == "3": list_contacts(contacts)
        elif choice == "4": update_contact(contacts)
        elif choice == "5": delete_contact(contacts)
        elif choice == "6":
            print("Khuda Hafiz!")
            break
        else:
            print("[Error] Galt choice! Dobara koshish karein.")

if __name__ == "__main__":
    main()



===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Galt choice! Dobara koshish karein.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Name khali nahi ho sakta!

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Galt choice! Dobara koshish karein.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Galt choice! Dobara koshish karein.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Galt choice! Dobara koshish karein.

===== CONTACT MANAGER =====
1. Add Contact
2. Search Contact
3. List All Contacts
4. Update Contact
5. Delete Contact
6. Exit
[Error] Galt choice! Dobara 

Task 15 | Integrated Order Fulfillment System

Concepts: OOP, composition, files, exceptions, algorithms

Problem: Design Product, Cart, Order, and Inventory classes. Read a product catalog from JSON, build carts, apply percentage discounts, compute totals, and place orders with stock updates. Persist orders to JSON.

Requirements: Reject invalid quantities and out-of-stock purchases. Validate all items before changing inventory; roll back if order persistence fails. Generate unique order IDs. Include at least three unit tests using unittest.

Sample / expected behavior: Catalog: P1 price=100 stock=5; cart P1 x2 with 10% discount -> total 180, stock becomes 3 after successful order.

In [ ]:
import json
import os
import uuid
from typing import Dict, List


# ==========================================
# CUSTOM EXCEPTIONS
# ==========================================
class InventoryError(Exception):
    """Raised when stock is insufficient or invalid."""
    pass


class ValidationError(Exception):
    """Raised when quantities or prices are invalid."""
    pass


class OrderPersistenceError(Exception):
    """Raised when writing the order to disk fails."""
    pass


# ==========================================
# CORE CLASSES
# ==========================================
class Product:
    def __init__(self, product_id: str, name: str, price: float, stock: int):
        if price < 0:
            raise ValidationError("Price cannot be negative.")
        if stock < 0:
            raise ValidationError("Initial stock cannot be negative.")
        self.product_id = product_id
        self.name = name
        self.price = price
        self.stock = stock

    def to_dict(self) -> dict:
        return {
            "product_id": self.product_id,
            "name": self.name,
            "price": self.price,
            "stock": self.stock
        }


class Inventory:
    def __init__(self):
        self.products: Dict[str, Product] = {}

    def load_from_json(self, filepath: str):
        """Loads product catalog from a JSON file."""
        if not os.path.exists(filepath):
            return
        with open(filepath, 'r') as f:
            data = json.load(f)
            for item in data:
                prod = Product(item['product_id'], item['name'], item['price'], item['stock'])
                self.products[prod.product_id] = prod

    def save_to_json(self, filepath: str):
        """Saves current inventory back to JSON."""
        with open(filepath, 'w') as f:
            json.dump([p.to_dict() for p in self.products.values()], f, indent=4)

    def add_product(self, product: Product):
        self.products[product.product_id] = product

    def validate_and_deduct(self, items: Dict[str, int]) -> Dict[str, int]:
        """
        Validates all items first. If all are valid, deducts stock.
        Returns a dictionary of original stock levels for rolling back if needed.
        """
        # Step 1: Validate Phase
        for prod_id, qty in items.items():
            if qty <= 0:
                raise ValidationError(f"Invalid quantity {qty} for product {prod_id}.")
            if prod_id not in self.products:
                raise InventoryError(f"Product {prod_id} not found in inventory.")
            if self.products[prod_id].stock < qty:
                raise InventoryError(f"Insufficient stock for product {prod_id}. Requested: {qty}, Available: {self.products[prod_id].stock}")

        # Step 2: Commit Phase (Track original stock for potential rollback)
        original_stock = {}
        for prod_id, qty in items.items():
            original_stock[prod_id] = self.products[prod_id].stock
            self.products[prod_id].stock -= qty
        
        return original_stock

    def rollback_stock(self, original_stock: Dict[str, int]):
        """Rolls back stock to a previous state if persistence fails."""
        for prod_id, old_qty in original_stock.items():
            if prod_id in self.products:
                self.products[prod_id].stock = old_qty


class Cart:
    def __init__(self):
        self.items: Dict[str, int] = {}  # product_id -> quantity

    def add_item(self, product_id: str, quantity: int):
        if quantity <= 0:
            raise ValidationError("Quantity to add must be greater than zero.")
        self.items[product_id] = self.items.get(product_id, 0) + quantity

    def calculate_total(self, inventory: Inventory, discount_percentage: float = 0.0) -> float:
        if not 0 <= discount_percentage <= 100:
            raise ValidationError("Discount percentage must be between 0 and 100.")
        
        total = 0.0
        for prod_id, qty in self.items.items():
            if prod_id not in inventory.products:
                raise InventoryError(f"Product {prod_id} in cart does not exist in catalog.")
            total += inventory.products[prod_id].price * qty
            
        return total * (1 - (discount_percentage / 100))


class Order:
    def __init__(self, order_id: str, items: Dict[str, int], total_paid: float):
        self.order_id = order_id
        self.items = items
        self.total_paid = total_paid

    def to_dict(self) -> dict:
        return {
            "order_id": self.order_id,
            "items": self.items,
            "total_paid": self.total_paid
        }


# ==========================================
# SYSTEM ENGINE / ORDER FULFILLMENT
# ==========================================
class OrderFulfillmentSystem:
    def __init__(self, inventory_file: str, orders_file: str):
        self.inventory_file = inventory_file
        self.orders_file = orders_file
        self.inventory = Inventory()
        self.inventory.load_from_json(inventory_file)

    def place_order(self, cart: Cart, discount_percentage: float = 0.0) -> str:
        if not cart.items:
            raise ValidationError("Cannot place an order with an empty cart.")

        # Calculate final price
        total_paid = cart.calculate_total(self.inventory, discount_percentage)
        
        # Phase 1: Validate and update internal inventory memory
        original_stock = self.inventory.validate_and_deduct(cart.items)
        
        # Generate Order details
        order_id = f"ORD-{uuid.uuid4().hex[:8].upper()}"
        new_order = Order(order_id, cart.items.copy(), total_paid)

        # Phase 2: Persist changes to disk (Filesystem Transaction)
        try:
            # Append order to log
            orders_data = []
            if os.path.exists(self.orders_file):
                with open(self.orders_file, 'r') as f:
                    try:
                        orders_data = json.load(f)
                    except json.JSONDecodeError:
                        orders_data = []
            
            orders_data.append(new_order.to_dict())
            
            with open(self.orders_file, 'w') as f:
                json.dump(orders_data, f, indent=4)
                
            # Commit the catalog update to file
            self.inventory.save_to_json(self.inventory_file)
            
        except Exception as e:
            # Transaction failed! Roll back internal memory tracking.
            self.inventory.rollback_stock(original_stock)
            raise OrderPersistenceError(f"Critical System Failure saving transaction: {e}")

        return order_id


# ==========================================
# UNIT TESTS
# ==========================================
import unittest

class TestOrderFulfillmentSystem(unittest.TestCase):
    def setUp(self):
        self.inv_file = "test_inventory.json"
        self.ord_file = "test_orders.json"
        
        # Setup initial catalog data
        catalog = [
            {"product_id": "P1", "name": "Widget A", "price": 100.0, "stock": 5},
            {"product_id": "P2", "name": "Widget B", "price": 50.0, "stock": 2}
        ]
        with open(self.inv_file, 'w') as f:
            json.dump(catalog, f)
            
        self.system = OrderFulfillmentSystem(self.inv_file, self.ord_file)

    def tearDown(self):
        for file in [self.inv_file, self.ord_file]:
            if os.path.exists(file):
                os.remove(file)

    def test_successful_order_with_discount(self):
        """Validates successfully buying P1 x2 with 10% discount."""
        cart = Cart()
        cart.add_item("P1", 2)
        
        order_id = self.system.place_order(cart, discount_percentage=10.0)
        
        self.assertTrue(order_id.startswith("ORD-"))
        self.assertEqual(self.system.inventory.products["P1"].stock, 3)
        
        # Verify saved state
        with open(self.ord_file, 'r') as f:
            orders = json.load(f)
            self.assertEqual(orders[0]["total_paid"], 180.0)

    def test_reject_out_of_stock(self):
        """Validates rejection when ordering more than available quantity."""
        cart = Cart()
        cart.add_item("P2", 3)  # Only 2 in stock
        
        with self.assertRaises(InventoryError):
            self.system.place_order(cart)
            
        # Confirm stock didn't change
        self.assertEqual(self.system.inventory.products["P2"].stock, 2)

    def test_rollback_on_persistence_failure(self):
        """Simulates file save failure and ensures stock is rolled back."""
        cart = Cart()
        cart.add_item("P1", 2)
        
        # Intentionally break pathing during execution to trigger file save failure
        self.system.orders_file = "/invalid_directory/non_existent_file.json"
        
        with self.assertRaises(OrderPersistenceError):
            self.system.place_order(cart)
            
        # Ensure stock rolled back to 5 from 3
        self.assertEqual(self.system.inventory.products["P1"].stock, 5)


if __name__ == "__main__":
    unittest.main()


usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [--durations N] [-f]
                             [-c] [-b] [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument 'c:\\Users\\HP\\AppData\\Roaming\\jupyter\\runtime\\kernel-v32e752870ff922406d1d183f708c38c3cc35bcdf4.json'


SystemExit: 2